In [1]:
import os
import json
import time
from pathlib import Path

import torch
import pandas as pd
import numpy as np

from tqdm.auto import tqdm


from datasets import Dataset


from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    DataCollatorForSeq2Seq,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer
)


from peft import (
    LoraConfig,
    get_peft_model,
    TaskType
)

D:\dev\projects\fourlang_translation\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
import sys

print(sys.executable)

D:\dev\projects\fourlang_translation\.venv\Scripts\python.exe


In [3]:
device=torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)


print(device)


if torch.cuda.is_available():
    print(
        torch.cuda.get_device_name(0)
    )

cuda
NVIDIA GeForce RTX 5060 Laptop GPU


In [4]:
PROJECT_ROOT = Path(
    r"D:\dev\projects\fourlang_translation"
)


DATA_DIR = (
    PROJECT_ROOT
    /
    "data"
    /
    "clean"
    /
    "en_uz"
    /
    "exp1"
)



OUTPUT_DIR = (
    PROJECT_ROOT
    /
    "models"
    /
    "lora"
    /
    "exp1_small100_uz"
)


OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

In [5]:
train_df=pd.read_json(
    DATA_DIR/"train.jsonl",
    lines=True
)


valid_df=pd.read_json(
    DATA_DIR/"validation.jsonl",
    lines=True
)


test_df=pd.read_json(
    DATA_DIR/"test.jsonl",
    lines=True
)


print(train_df.shape)
print(valid_df.shape)
print(test_df.shape)

(16000, 4)
(2000, 4)
(2000, 4)


In [6]:
print(
    train_df.head()
)

  src_lang tgt_lang                                           src_text  \
0       uz       en  Ҳеч шубҳа йўқки, уларга, албатта, дўзах бўлур ...   
1       en       uz                                  Remember _forever   
2       en       uz  So do not speak too softly, lest the sick at h...   
3       en       uz  Then He will return you into it and extract yo...   
4       uz       en  Ёки Мусонинг саҳифаларидаги сўзлар хабари унга...   

                                            tgt_text  
0             (Tafsir Al-Qurtubi, Vol. 10, Page 121)  
1                                 _Doim eslab qolish  
2  Овозингизни майин, назокатли қилиб, эркак киши...  
3      Сўнгра сизларни унга қайтариб, яна чиқарадир.  
4  Has he not been informed of what is in the Scr...  


In [7]:
train_dataset=Dataset.from_pandas(
    train_df,
    preserve_index=False
)


valid_dataset=Dataset.from_pandas(
    valid_df,
    preserve_index=False
)

In [8]:
MODEL_NAME="alirezamsh/small100"


tokenizer=AutoTokenizer.from_pretrained(
    MODEL_NAME
)


model=AutoModelForSeq2SeqLM.from_pretrained(
    MODEL_NAME
)


model.to(device)

M2M100ForConditionalGeneration(
  (model): M2M100Model(
    (shared): M2M100ScaledWordEmbedding(128112, 1024, padding_idx=1)
    (encoder): M2M100Encoder(
      (embed_tokens): M2M100ScaledWordEmbedding(128112, 1024, padding_idx=1)
      (embed_positions): M2M100SinusoidalPositionalEmbedding()
      (layers): ModuleList(
        (0-11): 12 x M2M100EncoderLayer(
          (self_attn): M2M100SdpaAttention(
            (k_proj): Linear(in_features=1024, out_features=1024, bias=True)
            (v_proj): Linear(in_features=1024, out_features=1024, bias=True)
            (q_proj): Linear(in_features=1024, out_features=1024, bias=True)
            (out_proj): Linear(in_features=1024, out_features=1024, bias=True)
          )
          (self_attn_layer_norm): LayerNorm((1024,), eps=1e-05, elementwise_affine=True, bias=True)
          (activation_fn): ReLU()
          (fc1): Linear(in_features=1024, out_features=4096, bias=True)
          (fc2): Linear(in_features=4096, out_features=1024, bia

In [9]:
print(
    "uz:",
    tokenizer.get_lang_id("uz")
)


print(
    "en:",
    tokenizer.get_lang_id("en")
)

uz: 128096
en: 128022


In [10]:
lora_config=LoraConfig(

    task_type=TaskType.SEQ_2_SEQ_LM,


    r=8,


    lora_alpha=16,


    target_modules=[
        "q_proj",
        "v_proj"
    ],


    lora_dropout=0.05,


    bias="none"
)

In [11]:
model=get_peft_model(
    model,
    lora_config
)


model.print_trainable_parameters()

trainable params: 589,824 || all params: 333,325,312 || trainable%: 0.1770


In [12]:
MAX_LENGTH=128


def preprocess(example):


    tokenizer.src_lang=(
        example["src_lang"]
    )


    tokenizer.tgt_lang=(
        example["tgt_lang"]
    )


    output=tokenizer(

        example["src_text"],

        text_target=
        example["tgt_text"],

        max_length=
        MAX_LENGTH,

        truncation=True
    )


    return {

        "input_ids":
        output["input_ids"],

        "attention_mask":
        output["attention_mask"],

        "labels":
        output["labels"]

    }

In [13]:
token_train=train_dataset.map(
    preprocess,
    remove_columns=
    train_dataset.column_names
)


token_valid=valid_dataset.map(
    preprocess,
    remove_columns=
    valid_dataset.column_names
)

Map: 100%|██████████| 2000/2000 [00:00<00:00, 5842.19 examples/s]


In [14]:
collator=DataCollatorForSeq2Seq(
    tokenizer,
    model=model
)

In [15]:
import transformers

print(transformers.__version__)

4.46.3


In [16]:
training_args = Seq2SeqTrainingArguments(

    output_dir=str(OUTPUT_DIR),

    num_train_epochs=3,

    per_device_train_batch_size=2,

    per_device_eval_batch_size=2,

    gradient_accumulation_steps=8,

    learning_rate=2e-4,


    # 如果你的GPU支持
    fp16=torch.cuda.is_available(),


    logging_steps=50,


    eval_strategy="steps",

    eval_steps=500,


    save_strategy="steps",

    save_steps=500,


    save_total_limit=2,


    predict_with_generate=True,


    report_to="none"
)

In [17]:
trainer=Seq2SeqTrainer(

    model=model,

    args=training_args,

    train_dataset=token_train,

    eval_dataset=token_valid,

    tokenizer=tokenizer,

    data_collator=collator
)

C:\Users\WingYouther\AppData\Local\Temp\ipykernel_31184\1392857948.py:1: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Seq2SeqTrainer.__init__`. Use `processing_class` instead.
  trainer=Seq2SeqTrainer(


In [18]:
trainer.train()

Step,Training Loss,Validation Loss
500,4.529200,4.286522
1000,4.369300,4.062615
1500,4.254400,3.941961
2000,4.205200,3.873178
2500,4.112800,3.830268
3000,4.167500,3.812957


TrainOutput(global_step=3000, training_loss=4.390728271484375, metrics={'train_runtime': 1090.4599, 'train_samples_per_second': 44.018, 'train_steps_per_second': 2.751, 'total_flos': 2770942131535872.0, 'train_loss': 4.390728271484375, 'epoch': 3.0})

In [19]:
model.save_pretrained(
    OUTPUT_DIR
)


tokenizer.save_pretrained(
    OUTPUT_DIR
)


print(
    "saved"
)

saved


In [20]:
from peft import PeftModel


base_model=AutoModelForSeq2SeqLM.from_pretrained(
    MODEL_NAME
)


model=PeftModel.from_pretrained(
    base_model,
    OUTPUT_DIR
)


model.to(device)

model.eval()

PeftModelForSeq2SeqLM(
  (base_model): LoraModel(
    (model): M2M100ForConditionalGeneration(
      (model): M2M100Model(
        (shared): M2M100ScaledWordEmbedding(128112, 1024, padding_idx=1)
        (encoder): M2M100Encoder(
          (embed_tokens): M2M100ScaledWordEmbedding(128112, 1024, padding_idx=1)
          (embed_positions): M2M100SinusoidalPositionalEmbedding()
          (layers): ModuleList(
            (0-11): 12 x M2M100EncoderLayer(
              (self_attn): M2M100SdpaAttention(
                (k_proj): Linear(in_features=1024, out_features=1024, bias=True)
                (v_proj): lora.Linear(
                  (base_layer): Linear(in_features=1024, out_features=1024, bias=True)
                  (lora_dropout): ModuleDict(
                    (default): Dropout(p=0.05, inplace=False)
                  )
                  (lora_A): ModuleDict(
                    (default): Linear(in_features=1024, out_features=8, bias=False)
                  )
                  

In [33]:
def translate(
    text,
    src_lang,
    tgt_lang
):

    tokenizer.src_lang = src_lang
    tokenizer.tgt_lang = tgt_lang


    encoded = tokenizer(
        text,
        return_tensors="pt"
    ).to(device)


    generated_tokens = model.generate(

        **encoded,

        forced_bos_token_id=
        tokenizer.lang_code_to_id[tgt_lang],

        max_length=128,

        num_beams=5
    )


    result = tokenizer.decode(
        generated_tokens[0],
        skip_special_tokens=True
    )


    return result

In [34]:
print(
    translate(
        "Men talabaman.",
        "uz",
        "en"
    )
)

Sign up.


In [35]:
print(
    translate(
        "I am a student.",
        "en",
        "uz"
    )
)

Мен ҳиииииииииииииииииииииииииииииииииииииииииииииииииииииииииииииииииииииииииииииииииииииииииииииииииииииииииииииииииииииииииии


In [36]:
model.print_trainable_parameters()

trainable params: 0 || all params: 333,325,312 || trainable%: 0.0000


In [37]:
base_model = AutoModelForSeq2SeqLM.from_pretrained(
    "alirezamsh/small100"
)

base_model.to(device)

M2M100ForConditionalGeneration(
  (model): M2M100Model(
    (shared): M2M100ScaledWordEmbedding(128112, 1024, padding_idx=1)
    (encoder): M2M100Encoder(
      (embed_tokens): M2M100ScaledWordEmbedding(128112, 1024, padding_idx=1)
      (embed_positions): M2M100SinusoidalPositionalEmbedding()
      (layers): ModuleList(
        (0-11): 12 x M2M100EncoderLayer(
          (self_attn): M2M100SdpaAttention(
            (k_proj): Linear(in_features=1024, out_features=1024, bias=True)
            (v_proj): Linear(in_features=1024, out_features=1024, bias=True)
            (q_proj): Linear(in_features=1024, out_features=1024, bias=True)
            (out_proj): Linear(in_features=1024, out_features=1024, bias=True)
          )
          (self_attn_layer_norm): LayerNorm((1024,), eps=1e-05, elementwise_affine=True, bias=True)
          (activation_fn): ReLU()
          (fc1): Linear(in_features=1024, out_features=4096, bias=True)
          (fc2): Linear(in_features=4096, out_features=1024, bia

In [39]:
print(model.peft_config)

{'default': LoraConfig(peft_type=<PeftType.LORA: 'LORA'>, auto_mapping=None, base_model_name_or_path='alirezamsh/small100', revision=None, task_type='SEQ_2_SEQ_LM', inference_mode=True, r=8, target_modules={'q_proj', 'v_proj'}, lora_alpha=16, lora_dropout=0.05, fan_in_fan_out=False, bias='none', use_rslora=False, modules_to_save=None, init_lora_weights=True, layers_to_transform=None, layers_pattern=None, rank_pattern={}, alpha_pattern={}, megatron_config=None, megatron_core='megatron.core', loftq_config={}, use_dora=False, layer_replication=None, runtime_config=LoraRuntimeConfig(ephemeral_gpu_offload=False))}


In [62]:
def base_translate(
    text,
    src_lang,
    tgt_lang
):

    base_tokenizer.src_lang = f"__{src_lang}__"


    inputs = base_tokenizer(
        text,
        return_tensors="pt"
    ).to(device)


    outputs = base_model.generate(
        **inputs,
        forced_bos_token_id=
        base_tokenizer.convert_tokens_to_ids(
            f"__{tgt_lang}__"
        ),
        max_length=128,
        num_beams=5
    )


    return base_tokenizer.decode(
        outputs[0],
        skip_special_tokens=True
    )

In [63]:
base_translate(
    "Men talabaman.",
    "uz",
    "en"
)

KeyError: '__uz__'

In [ ]:
print(base_model.config.decoder_start_token_id)

In [50]:
print(base_tokenizer.eos_token_id)

2


In [26]:
translate(
    "Men talabaman.",
    "uz",
    "en"
)

'Sign up.'

In [25]:
print(
translate(
    "Men talabaman.",
    "uz",
    "en"
)
)

Sign up.


In [27]:
print(
    "uz" in tokenizer.lang_code_to_id
)

print(
    "en" in tokenizer.lang_code_to_id
)

True
True


In [28]:
print(type(model))
print(model.config.model_type)

<class 'peft.peft_model.PeftModelForSeq2SeqLM'>
m2m_100


In [29]:
print(tokenizer.src_lang)
print(tokenizer.tgt_lang)

uz
uz


In [30]:
print(model.name_or_path)

alirezamsh/small100


In [31]:
print(
translate(
    "Men talabaman.",
    "uz",
    "en"
)
)

Sign up.


In [23]:
print(
translate(
    "I am a student.",
    "en",
    "uz"
)
)

Мен ҳиииииииииииииииииииииииииииииииииииииииииииииииииииииииииииииииииииииииииииииииииииииииииииииииииииииииииииииииииииииииииии


In [41]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM


BASE_MODEL = "alirezamsh/small100"


base_tokenizer = AutoTokenizer.from_pretrained(
    BASE_MODEL
)


base_model = AutoModelForSeq2SeqLM.from_pretrained(
    BASE_MODEL
)


base_model.to(device)

base_model.eval()

M2M100ForConditionalGeneration(
  (model): M2M100Model(
    (shared): M2M100ScaledWordEmbedding(128112, 1024, padding_idx=1)
    (encoder): M2M100Encoder(
      (embed_tokens): M2M100ScaledWordEmbedding(128112, 1024, padding_idx=1)
      (embed_positions): M2M100SinusoidalPositionalEmbedding()
      (layers): ModuleList(
        (0-11): 12 x M2M100EncoderLayer(
          (self_attn): M2M100SdpaAttention(
            (k_proj): Linear(in_features=1024, out_features=1024, bias=True)
            (v_proj): Linear(in_features=1024, out_features=1024, bias=True)
            (q_proj): Linear(in_features=1024, out_features=1024, bias=True)
            (out_proj): Linear(in_features=1024, out_features=1024, bias=True)
          )
          (self_attn_layer_norm): LayerNorm((1024,), eps=1e-05, elementwise_affine=True, bias=True)
          (activation_fn): ReLU()
          (fc1): Linear(in_features=1024, out_features=4096, bias=True)
          (fc2): Linear(in_features=4096, out_features=1024, bia

In [51]:
def base_translate(
    text,
    src_lang,
    tgt_lang
):

    base_tokenizer.src_lang = src_lang


    encoded = base_tokenizer(
        text,
        return_tensors="pt"
    ).to(device)


    generated_tokens = base_model.generate(
        **encoded,
        forced_bos_token_id=
        base_tokenizer.get_lang_id(tgt_lang),
        max_length=128,
        num_beams=5
    )


    return base_tokenizer.batch_decode(
        generated_tokens,
        skip_special_tokens=True
    )[0]

In [52]:
base_translate(
    "Men talabaman.",
    "uz",
    "en"
)

'bilan qilmaydi.'

In [53]:
print(base_tokenizer.src_lang)

uz


In [54]:
base_tokenizer.src_lang="uz"

print(
    base_tokenizer.convert_tokens_to_ids(
        "uz"
    )
)

1305


In [55]:
print(type(base_tokenizer))

<class 'transformers.models.m2m_100.tokenization_m2m_100.M2M100Tokenizer'>


In [57]:
from transformers import M2M100Tokenizer

M2M100Tokenizer.from_pretrained("alirezamsh/small100")

M2M100Tokenizer(name_or_path='alirezamsh/small100', vocab_size=128004, model_max_length=1024, is_fast=False, padding_side='right', truncation_side='right', special_tokens={'bos_token': '<s>', 'eos_token': '</s>', 'unk_token': '<unk>', 'sep_token': '</s>', 'pad_token': '<pad>', 'additional_special_tokens': ['__af__', '__am__', '__ar__', '__ast__', '__az__', '__ba__', '__be__', '__bg__', '__bn__', '__br__', '__bs__', '__ca__', '__ceb__', '__cs__', '__cy__', '__da__', '__de__', '__el__', '__en__', '__es__', '__et__', '__fa__', '__ff__', '__fi__', '__fr__', '__fy__', '__ga__', '__gd__', '__gl__', '__gu__', '__ha__', '__he__', '__hi__', '__hr__', '__ht__', '__hu__', '__hy__', '__id__', '__ig__', '__ilo__', '__is__', '__it__', '__ja__', '__jv__', '__ka__', '__kk__', '__km__', '__kn__', '__ko__', '__lb__', '__lg__', '__ln__', '__lo__', '__lt__', '__lv__', '__mg__', '__mk__', '__ml__', '__mn__', '__mr__', '__ms__', '__my__', '__ne__', '__nl__', '__no__', '__ns__', '__oc__', '__or__', '__pa__',

In [58]:
base_translate(
    "Men talabaman.",
    "uz",
    "en"
)

'bilan qilmaydi.'

In [44]:
print(tokenizer.lang_code_to_id["uz"])
print(tokenizer.lang_code_to_id["en"])

128096
128022


In [59]:
print(base_model.config)

M2M100Config {
  "_attn_implementation_autoset": true,
  "_name_or_path": "alirezamsh/small100",
  "activation_dropout": 0.0,
  "activation_function": "relu",
  "architectures": [
    "M2M100ForConditionalGeneration"
  ],
  "attention_dropout": 0.1,
  "bos_token_id": 0,
  "d_model": 1024,
  "decoder_attention_heads": 16,
  "decoder_ffn_dim": 4096,
  "decoder_layerdrop": 0.0,
  "decoder_layers": 3,
  "decoder_start_token_id": 2,
  "dropout": 0.1,
  "encoder_attention_heads": 16,
  "encoder_ffn_dim": 4096,
  "encoder_layerdrop": 0.0,
  "encoder_layers": 12,
  "eos_token_id": 2,
  "init_std": 0.02,
  "is_encoder_decoder": true,
  "max_length": 256,
  "max_position_embeddings": 1024,
  "model_type": "m2m_100",
  "num_beams": 5,
  "num_hidden_layers": 12,
  "pad_token_id": 1,
  "scale_embedding": true,
  "torch_dtype": "float32",
  "transformers_version": "4.46.3",
  "use_cache": true,
  "vocab_size": 128112
}



In [60]:
text = "Men talabaman."

base_tokenizer.src_lang = "uz"

tokens = base_tokenizer(
    text,
    return_tensors="pt"
)

print(tokens["input_ids"][0][:10])

tensor([128096,   1570,  39467,   1905,      5,      2])


In [61]:
base_tokenizer.src_lang="uz"

inputs = base_tokenizer(
    "Men talabaman.",
    return_tensors="pt"
).to(device)


outputs = base_model.generate(
    **inputs,
    forced_bos_token_id=base_tokenizer.get_lang_id("en"),
    max_length=20,
    num_beams=5,
    early_stopping=True
)


print(outputs[0])

print(
    base_tokenizer.decode(
        outputs[0],
        skip_special_tokens=False
    )
)

tensor([     2, 128022,   4342,      5,      2], device='cuda:0')
</s> __en__ bilan.</s>


In [45]:
print(type(base_tokenizer))
print(base_tokenizer.name_or_path)

<class 'transformers.models.m2m_100.tokenization_m2m_100.M2M100Tokenizer'>
alirezamsh/small100


In [46]:
base_tokenizer.src_lang="uz"
base_tokenizer.tgt_lang="en"

print(base_tokenizer.src_lang)
print(base_tokenizer.tgt_lang)

uz
en


In [64]:
from transformers import M2M100ForConditionalGeneration, M2M100Tokenizer


model_name="facebook/m2m100_418M"


test_tokenizer=M2M100Tokenizer.from_pretrained(
    model_name
)


test_model=M2M100ForConditionalGeneration.from_pretrained(
    model_name
).to(device)


def test_translate(text):

    test_tokenizer.src_lang="uz"

    inputs=test_tokenizer(
        text,
        return_tensors="pt"
    ).to(device)


    output=test_model.generate(
        **inputs,
        forced_bos_token_id=
        test_tokenizer.get_lang_id("en")
    )


    return test_tokenizer.decode(
        output[0],
        skip_special_tokens=True
    )


print(test_translate("Men talabaman."))

But it is.


In [66]:
import pandas as pd


df = pd.read_csv(
    "data/clean/en_uz/exp2/test.csv"
)


df.head()

FileNotFoundError: [Errno 2] No such file or directory: 'data/clean/en_uz/exp2/test.csv'